In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Krish\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Krish\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Krish\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
PRIORITY_TERMS = r'\b(dear|support|team|ni|hello|hope|well|customer|data|please|message|team|nan|could|would|assistance|problem|issue|failure|system|update)\b'
HTML_ARTIFACTS = r'\b(href|src|width|height|font|size|face|arial|sans|serif|color|border|style|nbsp|img|align|center|br|div|table|tr|td|span|strong|em)\b'

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text) # Remove URLs
    text = re.sub(r'<.*?>', '', text) # Remove tags
    text = re.sub(HTML_ARTIFACTS, '', text) # Remove attributes
    text = re.sub(r'[\r\n\t\a\b]+', ' ', text)
    text = re.sub(PRIORITY_TERMS, '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 1]
    return ' '.join(tokens)

In [ ]:
# Load Data
a = pd.read_csv('../raw_data_sets/aa_dataset-tickets-multi-lang-5-2-50-version.csv')
b = pd.read_csv('../raw_data_sets/dataset-tickets-multi-lang-4-20k.csv')
c = pd.read_csv('../raw_data_sets/email_dataset.csv')
d=pd.read_csv('../cleaned_dataset-folder/balanced_dataset.csv')

# Filter English
a = a[a['language'] == 'en']
b = b[b['language'] == 'en']

# Combine Tickets
base_cols = ['type', 'priority', 'body', 'subject']
tag_cols = [f'tag_{i}' for i in range(1, 9)]
req_cols = base_cols + tag_cols
a_sel = a[[col for col in req_cols if col in a.columns]]
b_sel = b[[col for col in req_cols if col in b.columns]]
tickets = pd.concat([a_sel, b_sel], ignore_index=True)

# Feedback Logic
def process_type_row(row):
    curr = row.get('type', '')
    if curr in ['Incident', 'Change']:
        tags = [str(row.get(f'tag_{i}', '')).lower() for i in range(1, 9)]
        if 'feedback' in tags:
            return 'Feedback'
    if curr == 'Problem':
        return 'Complaint'
    return curr

tickets['type'] = tickets.apply(process_type_row, axis=1)
valid_types = ['Request', 'Complaint', 'Feedback']
tickets = tickets[tickets['type'].isin(valid_types)].copy()

# Prepare Spam
c = c.rename(columns={'Label': 'type', 'Body': 'body', 'Subject': 'subject'})
c = c[c['type'] == 'spam'].copy()
c['priority'] = 'low'
for t in tag_cols: c[t] = np.nan

# Master DF
master = pd.concat([tickets[base_cols], c[base_cols]], ignore_index=True)
master.dropna(subset=['body', 'type', 'priority'], inplace=True)

# Balance
min_cnt = master['type'].value_counts().min()
balanced = master.groupby('type').apply(lambda x: x.sample(min_cnt, random_state=42)).reset_index(drop=True)
balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print('Data Loading Complete. Shape:', balanced.shape)
print(balanced['type'].value_counts())

Data Loading Complete. Shape: (5456, 4)
type
spam         1364
Request      1364
Feedback     1364
Complaint    1364
Name: count, dtype: int64


C:\Users\Krish\AppData\Local\Temp\ipykernel_10236\1700276717.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced = master.groupby('type').apply(lambda x: x.sample(min_cnt, random_state=42)).reset_index(drop=True)


In [4]:
print('Cleaning Subject and Body...')
balanced['body_cleaned'] = balanced['body'].apply(clean_text)
balanced['subject_cleaned'] = balanced['subject'].apply(clean_text)
print('Cleaning Complete!')

Cleaning Subject and Body...
Cleaning Complete!


In [5]:
def train_and_eval(X, y, name):
    print(f'\n--- Training {name} ---')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
    X_train_vec = tfidf.fit_transform(X_train)
    X_test_vec = tfidf.transform(X_test)

    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train_vec, y_train)

    preds = model.predict(X_test_vec)
    print(f'{name} Accuracy: {accuracy_score(y_test, preds):.4f}')
    return model, tfidf, X_test, y_test

In [6]:
print('=== URGENCY ENSEMBLE ===')
y_urgency = balanced['priority']

# Train Body
u_body_model, u_body_tfidf, X_test_u_body, y_test_u = train_and_eval(balanced['body_cleaned'], y_urgency, 'Urgency (Body)')

# Train Subject (Ensure same split by random_state)
u_subj_model, u_subj_tfidf, X_test_u_subj, _ = train_and_eval(balanced['subject_cleaned'], y_urgency, 'Urgency (Subject)')

# Ensemble Prediction (Soft Voting)
probs_body = u_body_model.predict_proba(u_body_tfidf.transform(X_test_u_body))
probs_subj = u_subj_model.predict_proba(u_subj_tfidf.transform(X_test_u_subj))
avg_probs = (probs_body + probs_subj) / 2
final_preds_idx = np.argmax(avg_probs, axis=1)
final_preds = u_body_model.classes_[final_preds_idx]

print('\n--- Ensemble Urgency Results ---')
print(classification_report(y_test_u, final_preds))

# Save Models
joblib.dump(u_body_model, '../models/urgency_body_model.pkl')
joblib.dump(u_body_tfidf, '../models/urgency_body_tfidf.pkl')
joblib.dump(u_subj_model, '../models/urgency_subject_model.pkl')
joblib.dump(u_subj_tfidf, '../models/urgency_subject_tfidf.pkl')
print('Urgency Models Saved!')

=== URGENCY ENSEMBLE ===

--- Training Urgency (Body) ---
Urgency (Body) Accuracy: 0.6163

--- Training Urgency (Subject) ---
Urgency (Subject) Accuracy: 0.5833

--- Ensemble Urgency Results ---
              precision    recall  f1-score   support

        high       0.48      0.62      0.54       291
         low       0.94      0.63      0.75       455
      medium       0.51      0.60      0.55       346

    accuracy                           0.62      1092
   macro avg       0.64      0.62      0.61      1092
weighted avg       0.68      0.62      0.63      1092

Urgency Models Saved!


In [7]:
print('=== TYPE CLASSIFICATION ENSEMBLE ===')
y_class = balanced['type']

# Train Body
c_body_model, c_body_tfidf, X_test_c_body, y_test_c = train_and_eval(balanced['body_cleaned'], y_class, 'Type (Body)')

# Train Subject
c_subj_model, c_subj_tfidf, X_test_c_subj, _ = train_and_eval(balanced['subject_cleaned'], y_class, 'Type (Subject)')

# Ensemble Prediction
probs_body = c_body_model.predict_proba(c_body_tfidf.transform(X_test_c_body))
probs_subj = c_subj_model.predict_proba(c_subj_tfidf.transform(X_test_c_subj))
avg_probs = (probs_body + probs_subj) / 2
final_preds_idx = np.argmax(avg_probs, axis=1)
final_preds = c_body_model.classes_[final_preds_idx]

print('\n--- Ensemble Type Results ---')
print(classification_report(y_test_c, final_preds))

# Save Models
joblib.dump(c_body_model, '../models/class_body_model.pkl')
joblib.dump(c_body_tfidf, '../models/class_body_tfidf.pkl')
joblib.dump(c_subj_model, '../models/class_subject_model.pkl')
joblib.dump(c_subj_tfidf, '../models/class_subject_tfidf.pkl')
print('Type Models Saved!')

=== TYPE CLASSIFICATION ENSEMBLE ===

--- Training Type (Body) ---
Type (Body) Accuracy: 0.8901

--- Training Type (Subject) ---
Type (Subject) Accuracy: 0.7537

--- Ensemble Type Results ---
              precision    recall  f1-score   support

   Complaint       0.80      0.85      0.82       273
    Feedback       0.83      0.74      0.78       273
     Request       0.95      1.00      0.97       273
        spam       1.00      1.00      1.00       273

    accuracy                           0.90      1092
   macro avg       0.90      0.90      0.90      1092
weighted avg       0.90      0.90      0.90      1092

Type Models Saved!


In [ ]:
# Ensure widgets are installed
!pip install ipywidgets
import ipywidgets as widgets
from IPython.display import display, clear_output

print('=== Email Classifier Interface ===')

# Widgets
w_subject = widgets.Text(description='Subject:', placeholder='Enter email subject...')
w_body = widgets.Textarea(description='Body:', placeholder='Enter email body...', layout=widgets.Layout(height='150px', width='600px'))
w_btn = widgets.Button(description='Classify', button_style='primary')
w_out = widgets.Output()

def on_click(b):
    with w_out:
        clear_output()
        subj_text = w_subject.value
        body_text = w_body.value
        
        if not body_text:
            print('Please enter some text in the Body field.')
            return
        
        # Rule-Based Urgency Heuristic
        text_combined = (str(subj_text) + ' ' + str(body_text)).lower()
        high_triggers = ['deadline', 'asap', 'immediate', 'urgency', 'critical', 'breach', 'emergency', 'shuts down', 'exploded', 'security alert', 'system down', 'outage', 'unacceptable']
        rule_triggered = False
        for word in high_triggers:
            if word in text_combined:
                urgency = 'high'
                u_conf = 0.99
                rule_triggered = True
                print(f'  [Rule Trigger] Found critical term: {word}')
                break
        
        # Urgency Prediction (ML) if no rule triggered
        if not rule_triggered:
            b_vec_u = u_body_tfidf.transform([clean_text(body_text)])
            s_vec_u = u_subj_tfidf.transform([clean_text(subj_text)])
            p_b_u = u_body_model.predict_proba(b_vec_u)[0]
            p_s_u = u_subj_model.predict_proba(s_vec_u)[0]
            avg_u = (p_b_u + p_s_u) / 2
            urgency = u_body_model.classes_[np.argmax(avg_u)]
            u_conf = np.max(avg_u)
        
        # Type Prediction
        b_vec_c = c_body_tfidf.transform([clean_text(body_text)])
        s_vec_c = c_subj_tfidf.transform([clean_text(subj_text)])
        p_b_c = c_body_model.predict_proba(b_vec_c)[0]
        p_s_c = c_subj_model.predict_proba(s_vec_c)[0]
        avg_c = (p_b_c + p_s_c) / 2
        e_type = c_body_model.classes_[np.argmax(avg_c)]
        t_conf = np.max(avg_c)
        
        # Display
        print(f'Subject: {subj_text}')
        print('-' * 40)
        print(f'Predicted Urgency: {urgency} ({u_conf:.1%})')
        print(f'Predicted Type:    {e_type} ({t_conf:.1%})')

w_btn.on_click(on_click)
display(w_subject, w_body, w_btn, w_out)

=== Email Classifier Interface ===



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Text(value='', description='Subject:', placeholder='Enter email subject...')

Textarea(value='', description='Body:', layout=Layout(height='150px', width='600px'), placeholder='Enter email…

Button(button_style='primary', description='Classify', style=ButtonStyle())

Output()

: 